In [ ]:
# COLAB only
!git clone https://github.com/ryszardtuora/kogni_2026.git
!python3 -m pip install -r kogni_2026/requirements.txt
%cd kogni_2026/laboratories/lab_5

# Data download
!git clone https://github.com/poleval/2024-emotion-recognition.git ../../data/2024-emotion-recognition
!wget http://dsmodels.nlp.ipipan.waw.pl/dsmodels/wiki-forms-all-100-skipg-ns-30-it100.txt.gz -O ../../data/others/wiki-forms-all-100-skipg-ns-30-it100.txt.gz
#!wget https://mozart.ipipan.waw.pl/~rtuora/wiki-forms-all-100-skipg-ns-30-it100.txt.gz -O ../../data/others/wiki-forms-all-100-skipg-ns-30-it100.txt.gz
!gunzip ../../data/others/wiki-forms-all-100-skipg-ns-30-it100.txt.gz

# Klasyfikacja tekstów

Na dzisiejszych tekstach nauczymy się korzystać z poznanych ostatnio metod reprezentacji tekstów w zadaniu klasyfikacji. Będziemy pracować również na zbiorze danych poznanym (pobieżnie) w trakcie poprzednich zajęć. Tym razem będzie nas interesować relacja między samymi tekstami, a przypisanymi im etykietami oznaczającymi ładunek emocjonalny.

Nasz dataset pochodzi z https://github.com/poleval/2024-emotion-recognition Wczytajmy dane, konkatenując (po kolumnach) informacje z dwu plików:

In [ ]:
import pandas
from pathlib import Path

DATA_DIR = Path("../../data")
EMO_DIR = DATA_DIR / "2024-emotion-recognition" / "train"
text_file = EMO_DIR / "in.tsv"
text_df = pandas.read_csv(text_file, sep="\t")
label_file = EMO_DIR / "expected.tsv"
label_df = pandas.read_csv(label_file, sep="\t")
emo_df = pandas.concat([text_df, label_df], axis=1)
emo_df

Spróbujmy spojrzeć na strukturę etykiet:

In [ ]:
emo_df[["Positive", "Negative"]].value_counts(normalize=True).sort_index()

Widzimy już pierwsze zaskakujące własności tego datasetu. Pracując z danymi zawsze warto zrobić sesję eksploracji danych i zweryfikować swoje aprioryczne założenia na temat ich struktury i własności. Cennym jest zapoznanie się z dokumentacją dostarczoną do danych co pozwoli odróżnić błędy i zanieczyszczenia w danych od celowych zabiegów twórców.

Przygotujmy klasyczny podział danych na trzy części:
1. train - służy do treningu pojedynczego modelu, z określoną kombinacją hiperparametrów
2. dev - służy do selekcji najlepszej kombinacji hiperparametrów, model nie ma dostępu do tych danych w trakcie treningu
3. test - służy do ewaluacji jakości wybranego modelu, te dane nie powinny być podstawą selekcji hiperparametrów

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, non_train_df = train_test_split(emo_df, test_size=0.2, random_state=SEED)
dev_df, test_df = train_test_split(non_train_df, test_size=0.5, random_state=SEED)

Spróbujmy przygotować reprezentację wektorową tekstów zaczynając od metody TF-IDF, wektoryzator trenujemy na podstawie danych treningowych, tak żeby informacje leksykalne również nie przedostały się do finalnego modelu:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_train_vectors = tfidf.fit_transform(train_df["text"])

Najpierw zajmiemy się treningiem klasyfikatora binarnego, i.e. będzie nas interesowała pojedyncza etykieta, np. `Positive`.

Skorzystamy z regresji logistycznej, która każdej z cech przypisuje współczynnik istotności, po czym ważona współczynnikami suma cech przekształcona logarytmicznie jest mapowana na prawdopodobieństwo:

In [ ]:
import numpy
from sklearn.linear_model import LogisticRegression

#target_label = "Positive"
target_label = "Joy"

train_labels = numpy.array(train_df[target_label])
log_reg = LogisticRegression(random_state=SEED)
_ = log_reg.fit(tfidf_train_vectors, train_labels)

Ogromną zaletą regresji logistycznej zastosowanej do reprezentacji leksykalnych jest prostota i interpretowalność - niektóre słowa po prostu "pchają" decyzję w stronę pozytywną:

In [ ]:
vocab = tfidf.get_feature_names_out()
coefficients = log_reg.coef_
coef_df = pandas.DataFrame({"coef": coefficients[0]}).reset_index()
coef_df["word"] = coef_df["index"].map(lambda x: vocab[x])
word_pres = ((tfidf_train_vectors > 0) * 1).mean(axis=0).A[0]
coef_df["presence"] = word_pres
coef_df.sort_values("coef").tail(20)

Inne słowa zaś w stronę negatywną:

In [ ]:
coef_df.sort_values("coef").head(20)

Zobaczmy jak sprawuje się model na dev secie:

In [ ]:
tfidf_dev_vectors = tfidf.transform(dev_df["text"])
dev_labels = numpy.array(dev_df[target_label])
dev_preds = log_reg.predict(tfidf_dev_vectors)
accuracy = (dev_preds == dev_labels).mean()
print(f"Accuracy: {accuracy * 100:.2f}%")

Przeważnie w naszej ewaluacji chcemy jednak skorzystać z bardziej wyrafinowanych metryk, które będą bardziej odporne na dysproporcje w dystrybucji etykiet. Klasyfikacja binarna nie jest symetryczna, fałszywy wynik pozytywny może okazać się dużo bardziej kosztowny niż fałszywy negatywny, i *vice versa*. Dlatego przeważnie korzysta się w takim zadaniu z:
1. `precision` - proporcji obiektów rozpoznanych jako pozytywne, które faktycznie są pozytywne (swoistość)
2. `recall` - proporcji pozytywnych obiektów, rozpoznanych jako pozytywne (czułość)
3. `f1` - średniej harmonicznej `precision` i `recall`

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

p, r, f1, _ = precision_recall_fscore_support(dev_labels, dev_preds, average="binary")
for lab, score in zip(["precision", "recall", "f1"], [p,r,f1]):
    print(f"{lab}: {round(score * 100, 2)}%")

W tym wypadku klasyfikator jest zbyt "liberalny" w przyznawaniu pozytywnych etykiet.

Regresja logistyczna dla każdej obserwacji przypisuje liczbową "siłę przekonania" - *confidence* względem etykiety pozytywnej, jeżeli jest ona wyższa od $0.5$, interpretujemy to jako wartość pozytywną. Zależność poprawności modelu od wartości *confidence* nazywamy kalibracją. Dobrze skalibrowany model to taki, którego pewność liniowo koreluje z proporcją poprawnie przyznanych etykiet, przeważnie przedstawia się to na wykresie, na którym dla decyzji pozytywnych klasyfikatora przedstawiamy jaka proporcja była prawidłowa w zależności od pewności modelu:

In [ ]:
from sklearn.calibration import calibration_curve
from matplotlib import pyplot as plt

NUM_BINS = 10
dev_probs = log_reg.predict_proba(tfidf_dev_vectors).T[1]
prob_true, prob_pred = calibration_curve(dev_labels, dev_probs, n_bins=NUM_BINS, strategy="uniform")
plt.plot(prob_pred, prob_true)
plt.plot((0,1), (0,1), c="red", linestyle="--")
plt.ylim(0,1)
plt.xlim(0, 1)
plt.ylabel("True frequency")
plt.xlabel("Confidence")
plt.show()

Poznaliśmy liczbowe wskaźniki jakości modelu, dalej jednak nie wiemy, czy świadczy to o tym, że nasz model jest dobry, czy marny. Na to ostatnie pytanie możemy sobie odpowiedzieć dopiero posiadając jakiś punkt referencyjny. Przygotujmy dwa takie *baseline*'y:
1. pierwszy będzie przypisywał etykiety losowo, z równym prawdopodobieństwem
2. drugi za każdym razem przypisze tę etykietę, która w danych treningowych okazała się najczęstsza

In [ ]:
import numpy
numpy.random.seed(SEED)

random_preds = numpy.random.random(dev_preds.shape) >= 0.5
most_common_label = train_df[target_label].astype(int).mode()[0]
freq_preds = numpy.ones(dev_preds.shape) * most_common_label
clf_eval = []
for clf_name, preds in [
    ("random", random_preds),
    ("freq", freq_preds),
    ("LR", dev_preds)
]:
    p, r, f1, _ = precision_recall_fscore_support(dev_labels, preds, average="binary")
    accuracy = accuracy_score(dev_labels, preds)
    clf_eval.append({"name": clf_name, "p":p, "r":r, "f1": f1, "acc": accuracy})

(pandas.DataFrame(clf_eval).set_index("name") * 100).round(2)

Widzimy, że klasyfikator nie jest wcale piorunująco lepszy od najprostszych technik generowania predykcji. To one wyznaczają naturalną "podłogę"
naszych oczekiwań. W wypadku realistycznych zbiorów danych sufit także istnieje, i jest wyznaczony przez jakość datasetu, to ile błędów może zawierać. Możemy czasem szacować taką liczbę podając wartości określające wewnętrzną spójność procesu anotacji. Mając do dyspozycji surowe dane, moglibyśmy obliczyć te wartości z pomocą wyspecjalizowanych metryk (np. *kappy Cohena*). W wypadku tego datasetu, nie mamy dostępu do takich informacji w wypadku etykiety "Joy". Pewnego rodzaju zastępnikiem takiej informacji dla innej etykiety - "Positive" jest jej relacja z etykietą "Negative":

In [ ]:
dev_df["Agreement"] = dev_df["Positive"] != dev_df["Negative"]
(dev_df["Agreement"].value_counts(normalize=True) * 100).round(2)

## Ćwiczenie 1:

Dla każdej wartości w zakresie $0, 0.01, 0.02 ... 0.99, 1$ oblicz jakie predykcje wydałby klasyfikator, przyjmując ją jako granicę decyzji. Następnie sprawdź jaka jest:
1. proporcja wyników fałszywie pozytywnych
2. proporcja wyników prawdziwie pozytywnych

Zbierz te dane, i przedstaw na wykresie jaka jest między nimi zależność, i.e. jak wzrost proporcji wynikow fałszywie pozytywnych idzie w parze z wzrostem proporcji prawdziwie pozytywnych

In [ ]:
# TODO

Chcemy jakoś usprawnić nasz klasyfikator. Na razie pozostawmy architekturę bez zmian, i zastanówmy się nad tym, ile informacji wyciągamy z samych danych. Zwróćmy uwagę, że reprezentacja TF-IDF traci informację o wielkości liter. To na pewno jest cecha która wiąże się z emocjami autorów:

In [ ]:
train_is_allcaps = train_df["text"].apply(lambda x: x == x.upper())
print("allcaps share:", train_is_allcaps.value_counts(normalize=True))

Aby nasz klasyfikator mógł skorzystać ze wskazanej cechy, wystarczy dokonkatenować ją (jako wektor) do istniejącej reprezentacji macierzowej danych. Napiszmy w tym celu funkcję pomocniczą:

In [ ]:
import scipy

def join_feat(vectors, feat):
    sparse_feat = scipy.sparse.csr_array([[val] for val in feat.tolist()])
    return scipy.sparse.hstack([vectors, sparse_feat])

aug_train_vectors = join_feat(tfidf_train_vectors, train_is_allcaps)
dev_is_allcaps = dev_df["text"].apply(lambda x: x == x.upper())
aug_dev_vectors = join_feat(tfidf_dev_vectors, dev_is_allcaps)
aug_train_vectors

Wytrenujmy klasyfikator i sprawdźmy wyniki:

In [ ]:
aug_log_reg = LogisticRegression(random_state=SEED)
_ = aug_log_reg.fit(aug_train_vectors, train_labels)
aug_preds = aug_log_reg.predict(aug_dev_vectors)
p, r, f1, _ = precision_recall_fscore_support(dev_labels, aug_preds, average="binary")
accuracy = accuracy_score(dev_labels, aug_preds)
clf_eval.append({"name": "aug_LR", "p":p, "r":r, "f1": f1, "acc": accuracy})
pandas.DataFrame(clf_eval).set_index("name")

## Ćwiczenie 2:

Sprawdź jaki współczynnik klasyfikator przypisał dodanej przez nas cesze? Czy uznał ją za wskazówkę pozytywną, czy negatywną?

In [ ]:
# TODO

## Ćwiczenie 3:

Na podstawie uzyskanych wyżej informacji o sposobie funkcjonowania modelu zaproponuj sposób, w jaki można zmienić tekst tak, aby wytrenowany klasyfikator zmienił decyzję z negatywnej na pozytywną.

In [ ]:
# TODO

## Ćwiczenie 4:

Spróbuj zaproponować inne cechy, które mogą pozytywnie wpłynąć na wyniki klasyfikatora. Przeprowadź eksperyment sprawdzający Twoją hipotezę

In [ ]:
# TODO

Spróbujmy zareprezentować nasze dane korzystając z wektoryzacji gęstej, używanymi ostatnio embeddingami:

In [ ]:
import re

EMB_DIM = 100

tokenizer = re.compile(r"\b\w+\b")
w2v_path = DATA_DIR / "others" / "wiki-forms-all-100-skipg-ns-30-it100.txt"
w2v = pandas.read_csv(w2v_path, sep=" ", skiprows=[0], header=None)
emb_arr = numpy.vstack(w2v.set_index(0).apply(numpy.array, axis=1))
ind2w = w2v[0].to_dict()
w2ind = {w: ind for ind, w in ind2w.items()}

def get_vec(word):
    ind = w2ind.get(word, None)
    if ind:
        return emb_arr[ind]
    return numpy.zeros((EMB_DIM))

def text_to_vec(text):
    tokens = tokenizer.findall(text)
    if not tokens:
        return numpy.zeros(EMB_DIM)
    vecs = [get_vec(tok.lower()) for tok in tokens]
    arr = numpy.vstack(vecs)
    mean_vec = arr.mean(axis=0)
    return mean_vec

dense_train_vectors = numpy.vstack([text_to_vec(text) for text in train_df["text"]])
dense_dev_vectors = numpy.vstack([text_to_vec(text) for text in dev_df["text"]])

Sprawdźmy jak zachowa się taki klasyfikator:

In [ ]:
dense_log_reg = LogisticRegression(random_state=SEED)
_ = dense_log_reg.fit(dense_train_vectors, train_labels)
dense_preds = dense_log_reg.predict(dense_dev_vectors)
p, r, f1, _ = precision_recall_fscore_support(dev_labels, dense_preds, average="binary")
accuracy = accuracy_score(dev_labels, dense_preds)
clf_eval.append({"name": "dense_emb_LR", "p":p, "r":r, "f1": f1, "acc": accuracy})
pandas.DataFrame(clf_eval).set_index("name")

## Klasyfikator neuronowy

Wypróbujmy też bardziej współczesną metodę klasyfikacji: sztuczną sieć neuronową. Skorzystamy z biblioteki `torch` (https://docs.pytorch.org) pozwalającej na wygodny dostęp do "wnętrza" modelu. Architektura, to prosty perceptron z warstwą ukrytą. Metoda `forward` zawiera całą logikę przetwarzania danych przez sieć:
1. przekształcenie liniowe (pierwsza warstwa - ukryta)
2. sigmoidalna funkcja aktywacji
3. przekształcenie liniowe (druga warstwa - klasyfikator)
4. sigmoidalna funkcja aktywacji


In [ ]:
import torch

class MLP(torch.nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.first_layer = torch.nn.Linear(EMB_DIM, hidden_dim)
        self.activation = torch.nn.Sigmoid()
        self.clf = torch.nn.Linear(hidden_dim, 1)
        self.decision = torch.nn.Sigmoid()

    def forward(self, inputs):
        first_out = self.activation(self.first_layer(inputs))
        result = self.decision(self.clf(first_out))
        return result


Zobaczmy jak przygotować wejście do sieci, i jak wygląda wyjście:

In [ ]:
dense_train_tensor = torch.tensor(dense_train_vectors, dtype=torch.float32)
dense_dev_tensor = torch.tensor(dense_dev_vectors, dtype=torch.float32)

nn_clf = MLP(10)
nn_clf.forward(dense_train_tensor)

Wytrenujmy naszą sieć. 

Skorzystamy z funkcji straty BCELoss, oraz optymalizacji Adam:

In [ ]:
nn_clf = MLP(10)
loss_fun = torch.nn.BCELoss()
optimizer = torch.optim.Adam(nn_clf.parameters(), lr=0.1)

labels_to_tensor = lambda labels: torch.tensor([labels], dtype=torch.float32).T
train_y = labels_to_tensor(train_labels)
dev_y = labels_to_tensor(dev_labels)

train_losses = []
dev_losses = []
train_accs = []
dev_accs = []
for epoch in range(100):
    train_result = nn_clf.forward(dense_train_tensor)
    train_loss = loss_fun(train_result, train_y)
    train_loss.backward()
    train_losses.append(train_loss.item())
    train_accs.append((((train_result > 0.5) * 1.0 == train_y) * 1.0).mean().item())
    
    optimizer.step()
    optimizer.zero_grad()
    with torch.no_grad():
        dev_result = nn_clf.forward(dense_dev_tensor)
        dev_loss = loss_fun(dev_result, dev_y)
        dev_losses.append(dev_loss.item())
        dev_accs.append((((dev_result > 0.5) * 1.0 == dev_y) * 1.0).mean().item())

Zapisawszy historię uczenia, możemy przedstawić ją na wykresie:

In [ ]:
history_df = pandas.DataFrame({"train_loss": train_losses, "dev_loss": dev_losses, "train_acc": train_accs, "dev_acc": dev_accs})
history_df.plot()

Również dla sieci neuronowej jesteśmy w stanie naszkicować krzywą kalibracji:

In [ ]:
with torch.no_grad():
    nn_dev_probs = nn_clf(dense_dev_tensor)

prob_true, prob_pred = calibration_curve(dev_labels, nn_dev_probs, n_bins=NUM_BINS, strategy="uniform")
plt.plot(prob_pred, prob_true)
plt.plot((0,1), (0,1), c="red", linestyle="--")
plt.ylim(0,1)
plt.xlim(0, 1)
plt.ylabel("True frequency")
plt.xlabel("Confidence")
plt.show()

Zobaczmy jak zachowa się nasza sieć na tle pozostałych klasyfikatorów:

In [ ]:
nn_dev_preds = nn_dev_probs >= 0.5
p, r, f1, _ = precision_recall_fscore_support(dev_labels, nn_dev_preds, average="binary")
accuracy = accuracy_score(dev_labels, nn_dev_preds)
clf_eval.append({"name": "MLP", "p":p, "r":r, "f1": f1, "acc": accuracy})
pandas.DataFrame(clf_eval).set_index("name")

In [ ]:
#TODO